# 02d — Tanda & Session Constraint Layer

This notebook contains `build_tanda()` and `SessionTracker.` These sit **after** the scoring step and enforce
Argentine Tango DJ rules:

**Hard constraints (always enforced, both modes):**
- All songs in a tanda share the same **style** (tango / vals / milonga)
- Tanda size: **4 songs for tango, 3 for milonga or vals**

**Convention mode adds:**
- Same **orchestra + singer** within a tanda
- Similar **decade** within a tanda
- Same (orchestra, singer) combo **doesn't repeat** across tandas in a session

**Flexible mode:**
- Style + tanda size enforced, but **orchestra + singer may be mixed**
- Decade and session-repeat rules are off
- TODO: define when mixing orchestra/singer "makes sense"

In [1]:
# --- Tanda & Session Constraints ---

import pandas as pd
import numpy as np
from dataclasses import dataclass, field


# Planning modes — match UI session_state["s_planning"]
PLANNING_CONVENTION = "convention"  # strict: same orch+singer+style+decade, no session repeats
PLANNING_FLEXIBLE   = "flexible"   # relaxed: same style only; orch+singer may mix (TBD)

# Hard rule — tanda size by style (enforced in ALL modes)
TANDA_SIZE = {"tango": 4, "vals": 3, "milonga": 3}
DEFAULT_TANDA_SIZE = 4

# Styles that are never part of a tanda
EXCLUDED_STYLES = {"cortina"}


@dataclass
class SessionTracker:
    """Tracks which (orchestra, singer) combos have been used in this session."""
    used_combos: set = field(default_factory=set)

    def mark_used(self, orchestra: str, singer: str):
        self.used_combos.add((orchestra, singer))

    def is_available(self, orchestra: str, singer: str) -> bool:
        return (orchestra, singer) not in self.used_combos

    def reset(self):
        self.used_combos.clear()

    def __repr__(self):
        return f'SessionTracker({len(self.used_combos)} combos used)'


def build_tanda(
    scores: pd.Series,
    df: pd.DataFrame,
    session: SessionTracker,
    style: str,
    planning_mode: str = PLANNING_CONVENTION,
) -> dict:
    """Build a tanda from scored songs, respecting DJ rules.

    Hard constraints (both modes):
      - Only songs matching the requested style (cortinas always excluded)
      - Tanda size: 4 for tango, 3 for vals/milonga

    Convention mode adds:
      - Same orchestra + singer within a tanda
      - Decade consistency
      - No (orchestra, singer) repeat across tandas in a session

    Flexible mode:
      - Only style + tanda size enforced
      - Top-N songs by score within the style, regardless of orchestra/singer
      - TODO: define "makes sense" criteria for mixing orchestra/singer

    Args:
        scores: pd.Series indexed like df, higher = better match
        df: DataFrame with 'orchestra', 'singer', 'style', 'decade', 'filename' columns
        session: SessionTracker to enforce no-repeat rule
        style: requested style ('tango', 'vals', or 'milonga')
        planning_mode: 'convention' (strict) or 'flexible' (relaxed)

    Returns:
        dict with 'songs', 'orchestra', 'singer', 'style', 'decade',
        'tanda_size', 'group_score', 'n_candidates', 'planning_mode', 'fallback_reason'
    """
    is_convention = (planning_mode == PLANNING_CONVENTION)
    tanda_size = TANDA_SIZE.get(style, DEFAULT_TANDA_SIZE)

    work = df[['filename', 'orchestra', 'singer', 'style', 'decade']].copy()
    work['score'] = scores

    # --- Hard: exclude cortinas, filter to requested style ---
    work = work[~work['style'].isin(EXCLUDED_STYLES)]
    work = work[work['style'] == style]

    if len(work) < 2:
        return {'songs': [], 'style': style, 'tanda_size': tanda_size,
                'fallback_reason': f'fewer than 2 songs with style={style}'}

    # ===== FLEXIBLE MODE: pick top songs within style =====
    if not is_convention:
        tanda_songs = work.nlargest(tanda_size, 'score')

        return {
            'orchestra': 'mixed',
            'singer': 'mixed',
            'style': style,
            'decade': tanda_songs['decade'].mode().iloc[0] if tanda_songs['decade'].notna().any() else None,
            'tanda_size': tanda_size,
            'group_score': round(float(tanda_songs['score'].mean()), 4),
            'n_candidates': int(len(work)),
            'planning_mode': planning_mode,
            'fallback_reason': None,
            'songs': [
                {'filename': r['filename'], 'score': round(float(r['score']), 4),
                 'orchestra': r['orchestra'], 'singer': r['singer']}
                for _, r in tanda_songs.iterrows()
            ],
        }

    # ===== CONVENTION MODE: group by (orchestra, singer) within style =====
    grouped = work.groupby(['orchestra', 'singer'])

    group_stats = []
    for (orch, sing), grp in grouped:
        if len(grp) < min(2, tanda_size):
            continue
        top_scores = grp['score'].nlargest(min(tanda_size, len(grp)))
        group_stats.append({
            'orchestra': orch,
            'singer': sing,
            'group_score': top_scores.mean(),
            'n_songs': len(grp),
        })

    if not group_stats:
        return {'songs': [], 'style': style, 'tanda_size': tanda_size,
                'fallback_reason': f'no (orchestra, singer) groups with >= 2 {style} songs'}

    candidates = pd.DataFrame(group_stats).sort_values('group_score', ascending=False)

    # --- Session filter ---
    fallback_reason = None
    available = candidates[
        candidates.apply(lambda r: session.is_available(r['orchestra'], r['singer']), axis=1)
    ]
    if len(available) == 0:
        available = candidates
        fallback_reason = 'all orchestra+singer combos already used in session, repeating best match'
    candidates = available

    # --- Pick best group ---
    best = candidates.iloc[0]
    orch, sing = best['orchestra'], best['singer']
    pool = work[(work['orchestra'] == orch) & (work['singer'] == sing)].copy()

    # --- Decade consistency ---
    decade = None
    if 'decade' in pool.columns and pool['decade'].notna().any():
        decade = pool['decade'].mode().iloc[0]
        decade_pool = pool[pool['decade'] == decade]
        if len(decade_pool) >= 2:
            pool = decade_pool
        else:
            decade = pool['decade'].mode().iloc[0]
            if fallback_reason is None:
                fallback_reason = f'not enough songs in {decade} for this combo, mixed decades'

    # --- Select top songs ---
    tanda_songs = pool.nlargest(tanda_size, 'score')

    # Mark as used
    session.mark_used(orch, sing)

    return {
        'orchestra': orch,
        'singer': sing,
        'style': style,
        'decade': decade or (tanda_songs['decade'].mode().iloc[0] if tanda_songs['decade'].notna().any() else None),
        'tanda_size': tanda_size,
        'group_score': round(float(best['group_score']), 4),
        'n_candidates': int(best['n_songs']),
        'planning_mode': planning_mode,
        'fallback_reason': fallback_reason,
        'songs': [
            {'filename': r['filename'], 'score': round(float(r['score']), 4)}
            for _, r in tanda_songs.iterrows()
        ],
    }


print('Tanda & session constraint functions defined.')

Tanda & session constraint functions defined.


## Quick Test

Uses random scores to verify the pipeline works before plugging in real method scores.

In [2]:
# --- Quick smoke test with random scores ---
import sys
sys.path.insert(0, '..')

from pathlib import Path
from atdj.config import PROCESSED_DIR

FEATURES_DIR = Path(PROCESSED_DIR) / 'features_exp'
df_merged = pd.read_pickle(FEATURES_DIR / 'df_merged.pkl')
print(f'Loaded: {df_merged.shape}')
print(f'Style distribution:\n{df_merged["style"].value_counts().to_string()}')

# Simulate scores
rng = np.random.RandomState(42)
fake_scores = pd.Series(rng.rand(len(df_merged)), index=df_merged.index)


def print_tanda(tanda, label=''):
    """Pretty-print a tanda result."""
    print(f'\n{label}{tanda["style"].upper()} | {tanda.get("orchestra", "?")} / {tanda.get("singer", "?")} ({tanda.get("decade")})')
    print(f'  Size: {tanda["tanda_size"]}, group score: {tanda.get("group_score")}, candidates: {tanda.get("n_candidates")}')
    if tanda.get('fallback_reason'):
        print(f'  WARNING: {tanda["fallback_reason"]}')
    for s in tanda.get('songs', []):
        extra = f'  [{s["orchestra"]} / {s["singer"]}]' if 'orchestra' in s else ''
        print(f'    {s["score"]:.4f}  {s["filename"]}{extra}')


# --- Convention mode: one tanda per style ---
print('\n=== CONVENTION MODE ===')
session_conv = SessionTracker()
for style in ['tango', 'vals', 'milonga']:
    tanda = build_tanda(fake_scores, df_merged, session_conv, style=style,
                        planning_mode="convention")
    print_tanda(tanda, label=f'{style.upper()} tanda: ')

print(f'\nSession state: {session_conv}')

# --- Flexible mode: one tanda per style ---
print('\n=== FLEXIBLE MODE ===')
session_flex = SessionTracker()
for style in ['tango', 'vals', 'milonga']:
    tanda = build_tanda(fake_scores, df_merged, session_flex, style=style,
                        planning_mode="flexible")
    print_tanda(tanda, label=f'{style.upper()} tanda: ')

print(f'\nSession state: {session_flex}')

Loaded: (294, 94)
Style distribution:
style
tango      216
vals        41
milonga     37

=== CONVENTION MODE ===

TANGO tanda: TANGO | Ricardo Tanturi / Enrique Campos (1940s)
  Size: 4, group score: 0.8499, candidates: 6
    0.9132  03 esta noche al pasar.mp3
    0.8671  04 la abandone y no sabia.mp3
    0.8094  01 asi se canta.mp3
    0.5113  03 cantor de barrio.mp3

VALS tanda: VALS | Francisco Canaro / Instrumental (1930s)
  Size: 3, group score: 0.9187, candidates: 5
    0.9395  11 desde el alma.mp3
    0.9219  11 vibraciones del alma.mp3
    0.8948  13 francia.mp3

MILONGA tanda: MILONGA | Anibal Troilo / Francisco Fiorentino (1940s)
  Size: 3, group score: 0.8368, candidates: 5
    0.9699  28 mano brava.mp3
    0.8324  29 soy un porteño.mp3
    0.7081  29 del tiempo guapo.mp3

Session state: SessionTracker(3 combos used)

=== FLEXIBLE MODE ===

TANGO tanda: TANGO | mixed / mixed (1940s)
  Size: 4, group score: 0.9801, candidates: 216
    0.9901  24 amurado.mp3  [Pedro Laurenz /

## Integration with Saved Results

Loads saved CLAP and CoT-A results, reconstructs scores, and builds tandas in both planning modes.

In [3]:
import json

# --- Load saved results ---
results_dir = FEATURES_DIR

ALL_RESULTS = {}
for name, fname in [('clap', 'clap_results.json'), ('cot_llm_a', 'cot_llm_a_results.json'), ('sbert_direction', 'sbert_direction_results.json')]:
    path = results_dir / fname
    if path.exists():
        with open(path, encoding='utf-8') as f:
            ALL_RESULTS[name] = json.load(f)
        print(f'Loaded {name}: {len(ALL_RESULTS[name])} results')
    else:
        print(f'{fname} not found, skipping')

print(f'\nAvailable methods: {list(ALL_RESULTS.keys())}')

Loaded clap: 8 results
Loaded cot_llm_a: 8 results
Loaded sbert_direction: 8 results

Available methods: ['clap', 'cot_llm_a', 'sbert_direction']


In [4]:
def scores_from_clap_result(result: dict, df: pd.DataFrame) -> pd.Series:
    """Reconstruct a full scores Series from a CLAP result's top/bottom song lists."""
    score_map = {}
    for song in result.get('top_k_songs', []) + result.get('bottom_k_songs', []):
        score_map[song['filename']] = song['score']
    # Songs not in top/bottom get score 0 (unknown — they weren't saved)
    return df['filename'].map(score_map).fillna(0.0)


def scores_from_cot_a_result(result: dict, df: pd.DataFrame) -> pd.Series:
    """Re-score all songs using saved feature_ranges from a CoT-A result."""
    feature_ranges = result.get('feature_ranges_used', {})
    scores = pd.Series(0.0, index=df.index)
    total_weight = 0.0

    for fname, spec in feature_ranges.items():
        if fname not in df.columns:
            continue
        col = df[fname]
        if col.dtype not in ['float64', 'float32', 'int64']:
            continue

        fmin = spec.get('min', col.min())
        fmax = spec.get('max', col.max())
        weight = spec.get('weight', 1.0)
        range_width = fmax - fmin if fmax > fmin else 1.0

        feature_score = pd.Series(0.0, index=df.index)
        inside = (col >= fmin) & (col <= fmax)
        feature_score[inside] = 1.0

        below = col < fmin
        if below.any():
            dist = (fmin - col[below]) / range_width
            feature_score[below] = np.maximum(0, 1.0 - dist)

        above = col > fmax
        if above.any():
            dist = (col[above] - fmax) / range_width
            feature_score[above] = np.maximum(0, 1.0 - dist)

        scores += feature_score * weight
        total_weight += weight

    if total_weight > 0:
        scores /= total_weight
    return scores


def scores_from_sbert_direction_result(result: dict, df: pd.DataFrame) -> pd.Series:
    """Re-score all songs using saved feature_ranges from an SBERT-direction result.

    Uses percentile rank + per-feature direction, weighted by relevance.
    Matches the scoring logic in 02b_method3_sbert_direction.
    """
    feature_ranges = result.get('feature_ranges_used', {})
    scores = pd.Series(0.0, index=df.index)
    total_weight = 0.0

    for fname, spec in feature_ranges.items():
        if fname not in df.columns:
            continue
        col = df[fname]
        if col.dtype not in ['float64', 'float32', 'int64']:
            continue

        # Percentile rank (0-1)
        pct = col.rank(pct=True)

        # Apply direction: 'higher_better' → percentile as-is, 'lower_better' → invert
        if spec.get('direction') == 'lower_better':
            feature_score = 1.0 - pct
        else:
            feature_score = pct

        weight = spec.get('relevance', 1.0)
        scores += feature_score * weight
        total_weight += weight

    if total_weight > 0:
        scores /= total_weight
    # Normalize to 0-1
    if scores.max() > scores.min():
        scores = (scores - scores.min()) / (scores.max() - scores.min())

    return scores


print('Score reconstruction functions defined.')

Score reconstruction functions defined.


In [5]:
# --- Build tandas from saved results ---

def style_from_prompt(prompt: str) -> str:
    """Extract style from a prompt string. Defaults to 'tango'."""
    p = prompt.lower()
    if 'milonga' in p:
        return 'milonga'
    if 'vals' in p:
        return 'vals'
    return 'tango'


def run_tanda_demo(method_name, results, score_fn, df, planning_mode):
    """Run build_tanda for each prompt result and print output."""
    print(f'\n{"="*60}')
    print(f'Method: {method_name}  |  Mode: {planning_mode}')
    print(f'{"="*60}')

    session = SessionTracker()
    for result in results:
        prompt = result["prompt"]
        style = style_from_prompt(prompt)
        scores = score_fn(result, df)
        tanda = build_tanda(scores, df, session, style=style, planning_mode=planning_mode)
        print(f'\nPrompt: "{prompt}" → style={style}')
        print_tanda(tanda)

    print(f'\nSession: {session}')


# Run for each available method x planning mode
for mode in ["convention", "flexible"]:
    if 'clap' in ALL_RESULTS:
        run_tanda_demo('CLAP', ALL_RESULTS['clap'], scores_from_clap_result, df_merged, mode)
    if 'cot_llm_a' in ALL_RESULTS:
        run_tanda_demo('CoT-LLM-A', ALL_RESULTS['cot_llm_a'], scores_from_cot_a_result, df_merged, mode)
    if 'sbert_direction' in ALL_RESULTS:
        run_tanda_demo('SBERT-Direction', ALL_RESULTS['sbert_direction'], scores_from_sbert_direction_result, df_merged, mode)


Method: CLAP  |  Mode: convention

Prompt: "energetic tango with strong rhythm for experienced dancers" → style=tango

TANGO | Pedro Laurenz / Juan Carlos Casas (1940s)
  Size: 4, group score: 0.092, candidates: 6
    0.3680  23 no me extraña.mp3
    0.0000  01 al verla pasar.mp3
    0.0000  24 amurado.mp3
    0.0000  12 es mejor perdonar.mp3

Prompt: "melancholic and slow, perfect for a late-night vals" → style=vals

VALS | Juan D'Arienzo / Instrumental (1930s)
  Size: 3, group score: 0.2489, candidates: 13
    0.0000  14 el aeroplano.mp3
    0.0000  11 no llores madre.mp3

Prompt: "bright and cheerful milonga with clear melody" → style=milonga

MILONGA | Francisco Canaro / Ernesto Famá (1930s)
  Size: 3, group score: 0.0053, candidates: 3
    0.0159  29 la milonga de buenos aires.mp3
    0.0000  28 milonga sentimental.mp3
    0.0000  26 no hay tierra como la mia.mp3

Prompt: "dramatic tango with heavy bandoneon and dark mood" → style=tango

TANGO | Carlos Di Sarli / Jorge Durán (194


Prompt: "smooth and relaxed, good for warming up the floor" → style=tango

TANGO | Osvaldo Fresedo / Roberto Ray (1930s)
  Size: 4, group score: 0.1719, candidates: 6
    0.3474  17 no quiero verte llorar.mp3
    0.0000  21 telon.mp3
    0.0000  04 sollozos.mp3



Prompt: "classic golden-age tango from the 40s, warm and nostalgic" → style=tango

TANGO | Miguel Caló / Raúl Berón (1940s)
  Size: 4, group score: 0.1581, candidates: 6
    0.3274  29 corazon no le hagas caso.mp3
    0.3048  22 tarareando.mp3
    0.0000  28 domingo a la noche.mp3
    0.0000  23 un crimen.mp3

Prompt: "a lively vals from the 50s with a strong orchestra" → style=vals

VALS | Ricardo Tanturi / Alberto Castillo (1940s)
  Size: 3, group score: 0.171, candidates: 6
    0.0000  13 marisabel.mp3
    0.0000  14 mi romance.mp3

Prompt: "need a cortina — something upbeat and non-tango to reset the floor" → style=tango

TANGO | Tipica Victor / Instrumental (1920s)
  Size: 4, group score: 0.1001, candidates: 6
    0.3112  23 dominio.mp3
    0.0894  08 pato.mp3
    0.0000  24 cardos.mp3
    0.0000  22 che papusa oi.mp3

Session: SessionTracker(8 combos used)

Method: CoT-LLM-A  |  Mode: convention

Prompt: "energetic tango with strong rhythm for experienced dancers" → style=tango



Prompt: "melancholic and slow, perfect for a late-night vals" → style=vals

VALS | Anibal Troilo / Francisco Fiorentino (1940s)
  Size: 3, group score: 0.8402, candidates: 5
    0.8605  14 acordandome de vos.mp3
    0.8474  12 temblando.mp3
    0.8128  11 pedacito de cielo.mp3

Prompt: "bright and cheerful milonga with clear melody" → style=milonga

MILONGA | Juan D'Arienzo / Alberto Echagüe (1950s)
  Size: 3, group score: 0.9422, candidates: 8
    0.9128  28 chaparron.mp3
    0.8648  28 me gusta bailar milonga.mp3

Prompt: "dramatic tango with heavy bandoneon and dark mood" → style=tango

TANGO | Angel D'Agostino / Angel Vargas (1940s)
  Size: 4, group score: 0.8393, candidates: 6
    0.8823  16 el cocherito.mp3
    0.8523  16 rosita la santiagueña.mp3
    0.8239  16 mi viejo barrio.mp3
    0.7989  23 mas solo que nunca.mp3



Prompt: "smooth and relaxed, good for warming up the floor" → style=tango

TANGO | Miguel Caló / Raúl Berón (1940s)
  Size: 4, group score: 0.9737, candidates: 6
    1.0000  29 corazon no le hagas caso.mp3
    0.9753  26 en tus ojos de cielo.mp3
    0.9639  28 domingo a la noche.mp3
    0.9555  23 un crimen.mp3



Prompt: "classic golden-age tango from the 40s, warm and nostalgic" → style=tango

TANGO | Carlos Di Sarli / Alberto Podestá (1940s)
  Size: 4, group score: 0.9973, candidates: 6
    1.0000  16 va a cantar un ruiseñor.mp3
    1.0000  08 volver a vernos.mp3
    1.0000  19 motivo sentimental.mp3
    0.9890  18 junto a tu corazon.mp3

Prompt: "a lively vals from the 50s with a strong orchestra" → style=vals

VALS | Juan D'Arienzo / Instrumental (1930s)
  Size: 3, group score: 0.9526, candidates: 13
    0.9479  11 no llores madre.mp3
    0.9386  14 el aeroplano.mp3

Prompt: "need a cortina — something upbeat and non-tango to reset the floor" → style=tango

TANGO | Alfredo De Angelis / Julio Martel (1940s)
  Size: 4, group score: 0.9047, candidates: 6
    0.9723  08 bajo belgrano.mp3
    0.9513  22 altar sin luz.mp3
    0.9371  24 el ciruja.mp3
    0.7581  23 va llegando gente al baile.mp3

Session: SessionTracker(8 combos used)

Method: SBERT-Direction  |  Mode: convention

Prompt: "energ


Prompt: "melancholic and slow, perfect for a late-night vals" → style=vals

VALS | Francisco Canaro / Instrumental (1930s)
  Size: 3, group score: 0.8777, candidates: 5
    1.0000  14 maria esther.mp3
    0.8435  14 corazon de oro.mp3
    0.7897  13 francia.mp3

Prompt: "bright and cheerful milonga with clear melody" → style=milonga

MILONGA | Anibal Troilo / Francisco Fiorentino (1940s)
  Size: 3, group score: 0.7236, candidates: 5
    0.9406  29 soy un porteño.mp3
    0.6840  28 mano brava.mp3
    0.5463  29 del tiempo guapo.mp3

Prompt: "dramatic tango with heavy bandoneon and dark mood" → style=tango

TANGO | Tipica Victor / Instrumental (1920s)
  Size: 4, group score: 0.8206, candidates: 6
    0.9569  22 che papusa oi.mp3
    0.8737  07 de mi barrio.mp3
    0.7799  24 cardos.mp3
    0.6720  21 el porteñito.mp3

Prompt: "smooth and relaxed, good for warming up the floor" → style=tango

TANGO | Osvaldo Pugliese / Instrumental (1950s)
  Size: 4, group score: 0.8434, candidates: 6
  


Prompt: "a lively vals from the 50s with a strong orchestra" → style=vals

VALS | Juan D'Arienzo / Instrumental (1930s)
  Size: 3, group score: 0.7865, candidates: 13
    0.6351  14 el aeroplano.mp3
    0.6344  11 no llores madre.mp3



Prompt: "need a cortina — something upbeat and non-tango to reset the floor" → style=tango

TANGO | Rodolfo Biagi / Instrumental (1940s)
  Size: 4, group score: 0.7366, candidates: 6
    0.8773  13 didi.mp3
    0.7560  11 el yaguaron.mp3
    0.7149  14 quejas de bandoneon.mp3
    0.5296  21 el estribo.mp3

Session: SessionTracker(8 combos used)

Method: CLAP  |  Mode: flexible

Prompt: "energetic tango with strong rhythm for experienced dancers" → style=tango

TANGO | mixed / mixed (1940s)
  Size: 4, group score: 0.2683, candidates: 216
    0.3680  23 no me extraña.mp3  [Pedro Laurenz / Juan Carlos Casas]
    0.3581  14 quejas de bandoneon.mp3  [Rodolfo Biagi / Instrumental]
    0.3471  22 llorar por una mujer.mp3  [Enrique Rodriguez / Armando Moreno]
    0.0000  18 pastora.mp3  [Alfredo De Angelis / Carlos Dante-Julio Martel]

Prompt: "melancholic and slow, perfect for a late-night vals" → style=vals

VALS | mixed / mixed (1940s)
  Size: 3, group score: 0.3667, candidates: 41
    0.3


Prompt: "dramatic tango with heavy bandoneon and dark mood" → style=tango

TANGO | mixed / mixed (1940s)
  Size: 4, group score: 0.8719, candidates: 216
    0.8823  16 el cocherito.mp3  [Angel D'Agostino / Angel Vargas]
    0.8731  06 nada mas que un corazon.mp3  [Osvaldo Pugliese / Roberto Chanel]
    0.8682  02 cafe de los angelitos.mp3  [Anibal Troilo / Alberto Marino]
    0.8640  03 esta noche al pasar.mp3  [Ricardo Tanturi / Enrique Campos]

Prompt: "smooth and relaxed, good for warming up the floor" → style=tango

TANGO | mixed / mixed (1940s)
  Size: 4, group score: 0.9993, candidates: 216
    1.0000  18 pastora.mp3  [Alfredo De Angelis / Carlos Dante-Julio Martel]
    1.0000  21 el pescante.mp3  [Lucio Demare / Raúl Berón]
    1.0000  29 corazon no le hagas caso.mp3  [Miguel Caló / Raúl Berón]
    0.9972  17 me estan sobrando las penas.mp3  [Anibal Troilo / Alberto Marino]

Prompt: "classic golden-age tango from the 40s, warm and nostalgic" → style=tango

TANGO | mixed / mixed